# P14′ amendment 1: trích nhãn lại theo pipeline chính thức đã sửa. Kaggle **GPU T4 ×2**, khoảng 7–9 giờ, chạy **1 lần**
Không sinh lại mô tả: dùng lại mô tả của 3 part đã chạy, chỉ sinh nốt **284 clip MER2025-FG** còn thiếu.
Sửa (chi tiết trong `explore/ovmer/p14_official.py`, AMENDMENT 1):
- cắt mô tả ở dòng đầu (Omni-3B tự bịa tiếp lượt hội thoại ở 81% mô tả);
- gọi trích nhãn và hậu xử lý đúng như toolkit chính thức;
- dịch nhãn tiếng Trung sang tiếng Anh bằng prompt chính thức.

**Input cần gắn (Add Input):**
1. Output của notebook P14′ PART 1 (`notebook02a74f1d00`), có `desc_1_0.jsonl`, `desc_1_1.jsonl` và `p14_items.jsonl`.
2. Một dataset riêng tư chứa `results.zip` và `results-2.zip` của PART 2 và 3. Kaggle tự giải nén zip khi tạo dataset.

Secret: **HF_TOKEN** (cộng **MER2026_ZIP_PW** nếu được báo thiếu). Internet **On**.

In [ ]:
KOUT, WORK = '/kaggle/working', '/tmp/p14'
MODEL, NSAMP, TOTAL_H = 'Qwen/Qwen2.5-Omni-3B', 16, 10.0
import os, glob, json, time, shutil, subprocess
T0 = time.time()
sh = lambda cmd, check=True: subprocess.run(cmd, shell=True, check=check)
os.makedirs(WORK, exist_ok=True)
sh(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme')
sh(f'rm -rf {WORK}/affectgpt && git clone -q --depth 1 https://github.com/zeroQiaoba/AffectGPT {WORK}/affectgpt')
print('code commit:', subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
sh('pip -q install -U huggingface_hub qwen-omni-utils bitsandbytes', check=False)
from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    raise SystemExit('HF_TOKEN secret is not attached to THIS notebook: Add-ons -> Secrets -> tick HF_TOKEN') from None
try:
    os.environ['MER2026_ZIP_PW'] = UserSecretsClient().get_secret('MER2026_ZIP_PW')   # never printed
except Exception:
    pass
os.environ['HF_HOME'] = f'{WORK}/hf'
import torch; print('GPUs:', torch.cuda.device_count())

In [ ]:
# collect the stored descriptions (any attached input; Kaggle unpacks zips) and the item list
found = {}
for p in glob.glob('/kaggle/input/**/*.jsonl', recursive=True):
    b = os.path.basename(p)
    if b.startswith('._'):
        continue
    if b in ('desc_1_0.jsonl', 'desc_1_1.jsonl', 'desc_2_0.jsonl', 'desc_2_1.jsonl', 'desc_3_0.jsonl', 'desc_3_1.jsonl', 'p14_items.jsonl'):
        found.setdefault(b, p)
print({k: sum(1 for _ in open(v)) for k, v in sorted(found.items())})
need = ['desc_1_0.jsonl', 'desc_1_1.jsonl', 'desc_2_0.jsonl', 'desc_2_1.jsonl', 'desc_3_0.jsonl', 'desc_3_1.jsonl', 'p14_items.jsonl']
miss = [b for b in need if b not in found]
if miss:
    raise SystemExit(f'missing inputs: {miss} (attach the PART 1 notebook output and the dataset with results.zip / results-2.zip)')
ITEMS = f'{KOUT}/p14_items.jsonl'; shutil.copy(found['p14_items.jsonl'], ITEMS)
for b in need[:-1]:
    shutil.copy(found[b], f'{KOUT}/{b}')   # desc_2_g.jsonl doubles as the resume file of shard g

In [ ]:
# media for the 284 MER2025-FG clips that have no description yet (same paths as in the first run)
from huggingface_hub import hf_hub_download, snapshot_download
Z26 = hf_hub_download(repo_id='MERChallenge/MER2026', repo_type='dataset', filename='video_7z/video_track2_train_human/video_split.zip',
                      local_dir=f'{WORK}/MER2026', token=os.environ['HF_TOKEN'])
subprocess.run(['python', f'{WORK}/icme/explore/ovmer/p13_prepare.py', '--mer26_only', '--affectgpt', f'{WORK}/affectgpt',
                '--mer26zip', Z26, '--work', f'{WORK}/media', '--out', f'{WORK}/mer26_items.jsonl'], check=True)
snapshot_download(MODEL); snapshot_download('Qwen/Qwen2.5-7B-Instruct')
items = [json.loads(l) for l in open(ITEMS)]
have = {json.loads(l)['key'] for g in range(2) for l in open(f'{KOUT}/desc_2_{g}.jsonl') if json.loads(l).get('ok')}
todo = [it for it in items if it['ds'] == 'MER2025-FG' and it['key'] not in have]
print('MER2025-FG: described', len(have), '| to describe', len(todo), '| media present', sum(os.path.exists(it['video']) for it in todo))

In [ ]:
def run_parallel(make_cmd, logname):
    procs = []
    for g in range(2):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        procs.append(subprocess.Popen(make_cmd(g), env=env, stdout=open(f'{KOUT}/{logname}_{g}.txt', 'w'), stderr=subprocess.STDOUT))
    while any(p.poll() is None for p in procs):
        time.sleep(300)
        for g in range(2):
            lines = [l for l in open(f'{KOUT}/{logname}_{g}.txt', errors='ignore') if 'clips,' in l or 'extracted' in l or 'fail' in l or 'Error' in l]
            print(f'{logname} GPU{g}:', lines[-1].strip() if lines else '(starting)', flush=True)
    print(logname, 'return codes:', [p.returncode for p in procs])
R = f'{WORK}/icme/explore/ovmer/p13_mbr.py'; O = f'{WORK}/icme/explore/ovmer/p14_official.py'
# 1) finish the MER2025-FG descriptions (resume: same item list and shards as the first run)
run_parallel(lambda g: ['python', R, 'run', '--items', ITEMS, '--out', f'{KOUT}/desc_2_{g}.jsonl', '--hours', '3.3', '--model', MODEL,
                        '--n_samples', str(NSAMP), '--shard', f'{g}/2', '--protocol', 'official', '--only_ds', 'MER2025-FG'], 'desc')
DESC = f'{KOUT}/desc_all.jsonl'; seen = set()
with open(DESC, 'w') as f:
    for part in (1, 2, 3):
        for g in range(2):
            for l in open(f'{KOUT}/desc_{part}_{g}.jsonl'):
                r = json.loads(l)
                if r.get('ok') and r['key'] not in seen:
                    seen.add(r['key']); f.write(l if l.endswith('\n') else l + '\n')
print('descriptions:', len(seen), 'of', len(items))

In [ ]:
# 2) amended label extraction for all clips (primary sets first), within the session budget
left = max(0.5, TOTAL_H - (time.time() - T0) / 3600); print(f'extraction budget {left:.2f} h')
run_parallel(lambda g: ['python', O, 'extract', '--descs', DESC, '--out', f'{KOUT}/labels_a1_{g}.jsonl', '--shard', f'{g}/2',
                        '--amend', '1', '--hours', f'{left:.2f}'], 'extract')
LAB = f'{KOUT}/p14_labels_amend1.jsonl'
with open(LAB, 'w') as f:
    for g in range(2):
        if os.path.exists(f'{KOUT}/labels_a1_{g}.jsonl'): f.write(open(f'{KOUT}/labels_a1_{g}.jsonl').read())
subprocess.run(['python', f'{WORK}/icme/explore/ovmer/p14o_score.py', '--items', ITEMS, '--labels', LAB,
                '--mapping', f'{WORK}/affectgpt/AffectGPT-RL/emotion_wheel/wheel_mapping.npz', '--report', f'{KOUT}/p14_report_amend1.md'], check=False)
print(open(f'{KOUT}/p14_report_amend1.md').read())